# FER2013 Baseline Fresh — Kaggle T4 Full Run

This notebook runs the full 3-seed, 224x224, 100-epoch baseline on a Kaggle T4 GPU.

**Prerequisites:**
1. Add the FER2013 dataset: expects `fer2013.csv` at `/kaggle/input/fer2013/fer2013.csv`
2. Upload the `baseline_fresh/` folder (src/, config.yaml, main.py) as a dataset or via upload
3. Enable GPU: Settings → Accelerator → GPU T4 x2

In [ ]:
# Install pinned dependencies
!pip install -q torch==2.3.1 torchvision==0.18.1 \
    numpy==1.26.4 pandas==2.2.2 scikit-learn==1.5.1 \
    matplotlib==3.9.1 seaborn==0.13.2 Pillow==10.4.0 \
    opencv-python-headless==4.10.0.84 PyYAML==6.0.1 tqdm==4.66.4

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
import os

# Paths — adjust if your dataset name differs
CSV_PATH = "/kaggle/input/fer2013/fer2013.csv"
assert os.path.exists(CSV_PATH), f"fer2013.csv not found at {CSV_PATH}. Check your dataset input."

# Working directory for outputs
OUT_DIR = "/kaggle/working/baseline_fresh"
os.makedirs(OUT_DIR, exist_ok=True)

# If src/ was uploaded as a separate dataset, copy it
SRC_INPUT = "/kaggle/input/baseline-fresh-src"  # adjust if different
if os.path.exists(SRC_INPUT):
    !cp -r {SRC_INPUT}/* /kaggle/working/
    print("Copied source files from dataset input.")

os.chdir("/kaggle/working")
print(f"Working dir: {os.getcwd()}")
print(f"CSV: {CSV_PATH}")

In [ ]:
# Full run: 3 seeds, 224x224, 100 epochs
!python3 main.py \
    --config config.yaml \
    --csv {CSV_PATH} \
    --resolution 224 \
    --epochs 100 \
    --seeds 42 123 456 \
    --batch-size 64 \
    --num-workers 2 \
    --output-dir {OUT_DIR}

In [ ]:
# Display results
import json
with open(f"{OUT_DIR}/metrics.json") as f:
    m = json.load(f)
print(f"Test Macro-F1: {m['aggregate']['test_macro_f1_mean']:.4f} ± {m['aggregate']['test_macro_f1_std']:.4f}")
print(f"Test Accuracy: {m['aggregate']['test_acc_mean']:.4f} ± {m['aggregate']['test_acc_std']:.4f}")
print(f"Best seed: {m['best_seed']}")
print("\nPer-class F1 (best seed):")
best = m['per_seed'][str(m['best_seed'])]
for cls, f1 in best['test_per_class_f1'].items():
    print(f"  {cls:<10}: {f1:.4f}")

In [ ]:
# Show all figures
from IPython.display import Image, display
import glob

for fig_path in sorted(glob.glob(f"{OUT_DIR}/figures/*.png")):
    print(f"\n--- {os.path.basename(fig_path)} ---")
    display(Image(filename=fig_path, width=700))

In [ ]:
# Verify feature shapes and non-negativity
import numpy as np
for split in ['train', 'val', 'test']:
    feats = np.load(f"{OUT_DIR}/features/{split}_features.npy")
    labels = np.load(f"{OUT_DIR}/features/{split}_labels.npy")
    print(f"{split}: features {feats.shape}, labels {labels.shape}, "
          f"min={feats.min():.4f}, non-negative={bool((feats >= 0).all())}")

In [ ]:
# Package outputs for download
!cd /kaggle/working && tar czf baseline_fresh_results.tar.gz baseline_fresh/
print("\nDownload: baseline_fresh_results.tar.gz from /kaggle/working/")